# D20 S3 A1 - NumPy CNN Feature Extractor
Implements valid cross-correlation, ReLU, non-overlapping max pooling, and an end-to-end digit classifier.


In [1]:
import numpy as np
from sklearn.datasets import load_digits

digits = load_digits()
images = digits.images
labels = digits.target

kernel = np.array([[1, 0, -1],
                   [2, 0, -2],
                   [1, 0, -1]], dtype=float)


In [2]:
def conv2d(image, kernel):
    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    H, W = image.shape
    kh, kw = kernel.shape
    out = np.empty((H - kh + 1, W - kw + 1), dtype=float)
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i + kh, j:j + kw] * kernel)
    return out


def relu(x):
    return np.maximum(np.asarray(x), 0)


def maxpool2d(x, size=2):
    x = np.asarray(x)
    H, W = x.shape
    out_h, out_w = H // size, W // size
    trimmed = x[:out_h * size, :out_w * size]
    blocks = trimmed.reshape(out_h, size, out_w, size)
    return blocks.max(axis=(1, 3))


def extract_features(image, kernel):
    convolved = conv2d(image, kernel)
    activated = relu(convolved)
    pooled = maxpool2d(activated, size=2)
    return pooled.ravel()


In [3]:
print(conv2d([[1,2,3],[4,5,6],[7,8,9]], [[1,0],[0,1]]))
print(relu([[-1, 2], [3, -4]]))
print(maxpool2d([[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]], 2))
f = extract_features(images[0], kernel)
print(f.shape, f.min() >= 0)


[[ 6.  8.]
 [12. 14.]]
[[0 2]
 [3 0]]
[[ 6  8]
 [14 16]]
(9,) True


In [4]:
# End-to-end classifier using a 3-kernel feature bank
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

kernels = [
    np.array([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=float),
    np.array([[1,2,1],[0,0,0],[-1,-2,-1]], dtype=float),
    np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=float),
]
X = np.array([np.concatenate([extract_features(image, k) for k in kernels]) for image in images])
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42, stratify=labels)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=42))
model.fit(X_train, y_train)
accuracy = accuracy_score(y_test, model.predict(X_test))
print('Classifier accuracy:', accuracy)


Classifier accuracy: 0.9444444444444444


In [5]:
# Checks corresponding to all 8 autograder requirements
expected_conv = np.array([[6., 8.], [12., 14.]])
assert np.array_equal(conv2d([[1,2,3],[4,5,6],[7,8,9]], [[1,0],[0,1]]), expected_conv)

ones_conv = conv2d(np.ones((5,5)), np.ones((3,3)))
assert ones_conv.shape == (3,3)
assert np.all(ones_conv == 9)

r = relu(np.array([[-1,2],[3,-4]]))
assert r.shape == (2,2)
assert np.array_equal(r, np.array([[0,2],[3,0]]))

pool_input = np.arange(1,17).reshape(4,4)
assert np.array_equal(maxpool2d(pool_input,2), np.array([[6,8],[14,16]]))

feat = extract_features(images[0], kernel)
assert feat.shape == (9,)
assert np.all(feat >= 0)

signed = np.array([[1,-2,3,4],[-5,6,-7,8],[9,-10,11,-12],[13,14,-15,16]], dtype=float)
k2 = np.array([[1,-1],[-1,1]], dtype=float)
manual = maxpool2d(relu(conv2d(signed,k2)),2).ravel()
assert np.array_equal(extract_features(signed,k2), manual)

assert accuracy >= 0.80

odd_pool = maxpool2d(np.arange(25).reshape(5,5),2)
assert odd_pool.shape == (2,2)
assert np.array_equal(odd_pool, np.array([[6,8],[16,18]]))

print('All 8 autograder-style checks passed.')


All 8 autograder-style checks passed.
